# Local STARE-PODS Demo — no AWS / no RDS

Mirrors `demo_reconstitute_hdf5_from_s3.py` but uses the **local filesystem** for zarr storage and **SQLite** for metadata. No cloud credentials needed.

**Workflow**
1. Ingest a GMI granule → zarr groups on local disk + SQLite metadata (STARE partition at **level 4**)
2. Find intersecting data for a bounding box via STARE SIDs + SQLite (level 4)
3. Load intersecting zarr chunks from disk
4. Reconstitute an HDF5 file (both S1 and S2 scans) from the level-4 zarr partitions
5. Compare the reconstituted structure with the original granule
6. Verify SQLite metadata

In [20]:
#import subprocess, sys
#subprocess.check_call([sys.executable, "-m", "pip", "install", "-e",
#                       "/Users/tonhai/workspace/Bayesics/StarePandas_par/stare_demo_add_contruct_parallel",
#                       "-q"])

In [21]:
import os
import sqlite3
import h5py
from starepandas.demo import LocalStarePodsDemo

## Configuration

Edit these paths and parameters before running.

In [22]:
# zarr store + SQLite DB live here
LOCAL_ROOT = "/tmp/stare_pods_local"

GRANULE_FILE = (
    "/Users/thatdaihaiton/Workspace/STARE/L1C_Data_Samples/GPM/2025/Jan_1_2/"
    "1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5"
)

# STARE partition level used for both ingestion and bbox → SIDs lookup.
# Level 4 creates larger (coarser) spatial chunks than level 10.
STARE_LEVEL = 4

# Bounding box filter — set to None to reconstitute the full granule,
# or e.g. (115, -30, 120, -25) to restrict to SW Australia / Perth.
BBOX = None   # (lon_min, lat_min, lon_max, lat_max) or None

DATASETS = ["GMI_S1", "GMI_S2"]

OUTPUT_HDF5 = "/tmp/reconsitution/gmi_local_reconstituted.h5"

# Set to True to wipe LOCAL_ROOT before each run.
# IMPORTANT: re-running without cleaning causes duplicate SQLite entries,
# which inflates the reconstituted HDF5 (e.g. 3× the expected scan count).
# Keep True unless you intentionally want to append more granules.
CLEAN_BEFORE_RUN = True

print(f"Granule    : {os.path.basename(GRANULE_FILE)}")
print(f"Datasets   : {DATASETS}")
print(f"BBox       : {BBOX}  (None = full granule)")
print(f"STARE level: {STARE_LEVEL}")
print(f"Local root : {LOCAL_ROOT}")
print(f"Clean first: {CLEAN_BEFORE_RUN}")

Granule    : 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5
Datasets   : ['GMI_S1', 'GMI_S2']
BBox       : None  (None = full granule)
STARE level: 4
Local root : /tmp/stare_pods_local
Clean first: True


## Step 1 — Ingest granule → local zarr + SQLite

In [23]:
import shutil

if CLEAN_BEFORE_RUN and os.path.exists(LOCAL_ROOT):
    shutil.rmtree(LOCAL_ROOT)
    print(f"Removed existing data at {LOCAL_ROOT}")
else:
    print(f"Skipping cleanup (CLEAN_BEFORE_RUN={CLEAN_BEFORE_RUN})")

Removed existing data at /tmp/stare_pods_local


In [24]:
%%time
import time
demo = LocalStarePodsDemo(local_root=LOCAL_ROOT)

local_paths = demo.ingest_granules(GRANULE_FILE, instrument='GMI', level=STARE_LEVEL)
print(f"Written {len(local_paths)} scan path(s).")
for p in local_paths:
    print(f"  {p}")

INFO:starepandas.demo:Found 1 GMI file(s)
INFO:starepandas.demo:Processing 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5
INFO:starepandas.demo:✓ Stored 1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B.HDF5 → /tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B
INFO:starepandas.demo:Ingested 2 zarr dataset(s)


Written 2 scan path(s).
  /tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B
  /tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20250101-S034347-E051659.061567.V07B
CPU times: user 21.3 s, sys: 16.3 s, total: 37.6 s
Wall time: 39.2 s


## Step 2 — Find intersecting data via STARE SIDs

In [16]:
if BBOX is not None:
    location_sids = demo.get_sids_for_bbox(*BBOX, level=STARE_LEVEL)
    print(f"Generated {len(location_sids)} SIDs for bbox {BBOX}")
else:
    location_sids = None
    print("No bbox filter — all groups will be loaded (full granule reconstitution)")

intersecting = demo.find_intersecting_data(location_sids, instruments=['GMI'])
print(f"Found {len(intersecting)} metadata row(s).")
intersecting[['Dataset', 'grouped_id', 'group_path']]

INFO:starepandas.demo:Loaded all 514 groups for GMI


No bbox filter — all groups will be loaded (full granule reconstitution)
Found 514 metadata row(s).


,Dataset,grouped_id,group_path
0,GMI_S2,2094173826727280644,/tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20...
1,GMI_S2,2118943624677818372,/tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20...
2,GMI_S2,2163979620951523332,/tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20...
3,GMI_S2,2193253018529431556,/tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20...
4,GMI_S2,2096425626540965892,/tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20...
...,...,...,...
509,GMI_S1,2127950823932559364,/tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20...
510,GMI_S1,2107684625609392132,/tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20...
511,GMI_S1,2112188225236762628,/tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20...
512,GMI_S1,2114440025050447876,/tmp/stare_pods_local/1C.GPM.GMI.XCAL2016-C.20...


## Step 3 — Load intersecting zarr chunks from disk

In [17]:
if not intersecting.empty:
    data_dict = demo.download_and_analyze(
        intersecting,
        instruments=list(intersecting['Dataset'].unique()),
    )
    for ds_name, sdf in data_dict.items():
        print(f"{ds_name}: {len(sdf)} rows, columns: {list(sdf.columns[:6])} …")
        display(sdf.head(3))
else:
    print("No intersecting chunks found.")
    data_dict = {}

INFO:starepandas.demo:✓ Combined 659243 rows for GMI_S2
INFO:starepandas.demo:✓ Combined 659243 rows for GMI_S1


GMI_S2: 659243 rows, columns: ['sunGlintAngle', 'sunLocalTime', 'Tc3', 'SCstatus_SCaltitude', 'incidenceAngleIndex1', 'Tc1'] …


,sunGlintAngle,sunLocalTime,Tc3,SCstatus_SCaltitude,incidenceAngleIndex1,Tc1,Quality,Tc2,SCstatus_SCorientation,lon,...,SCstatus_SClatitude,SCstatus_FractionalGranuleNumber,incidenceAngleIndex2,SCstatus_SClongitude,incidenceAngle,lat,incidenceAngleIndex3,incidenceAngleIndex4,timestamp,sids
0,88,6.415025,248.050003,451.328461,1,256.399994,0,251.600006,180,41.155605,...,-65.10363,61567.000075,1,43.339439,49.57,-60.956139,1,1,2025-01-01 03:43:47.516,2095643662640840171
1,89,6.408187,248.229996,451.328461,1,256.359985,0,250.559998,180,41.053024,...,-65.10363,61567.000075,1,43.339439,49.57,-60.967239,1,1,2025-01-01 03:43:47.516,2095644871184156395
2,89,6.401363,247.250000,451.328461,1,255.940002,0,250.279999,180,40.950649,...,-65.10363,61567.000075,1,43.339439,49.57,-60.978867,1,1,2025-01-01 03:43:47.516,2095644225044829739


GMI_S1: 659243 rows, columns: ['Tc9', 'Tc8', 'Tc1', 'sunGlintAngle', 'Quality', 'Tc7'] …


,Tc9,Tc8,Tc1,sunGlintAngle,Quality,Tc7,incidenceAngleIndex1,incidenceAngleIndex6,Tc6,sunLocalTime,...,SCstatus_SCorientation,lat,incidenceAngleIndex2,timestamp,incidenceAngleIndex3,Tc5,Tc2,incidenceAngleIndex4,SCstatus_SClongitude,sids
0,209.960007,242.470001,162.190002,90,0,149.389999,1,1,209.000000,6.399077,...,180,-60.416740,1,2025-01-01 03:43:47.516,1,206.300003,88.510002,1,43.338219,2096919498826664619
1,210.119995,242.339996,162.830002,91,0,149.539993,1,1,209.020004,6.391473,...,180,-60.429089,1,2025-01-01 03:43:47.516,1,206.479996,89.080002,1,43.338219,2096462762016624139
2,209.949997,242.100006,161.820007,91,0,149.779999,1,1,209.750000,6.383884,...,180,-60.442020,1,2025-01-01 03:43:47.516,1,206.729996,88.800003,1,43.338219,2096462641002782731


## Step 4 — Reconstitute HDF5 (S1 + S2)

In [ ]:
%%time
import time
granule_basename = os.path.splitext(os.path.basename(GRANULE_FILE))[0]

recon_path = demo.reconstitute_hdf5(
    dataset=DATASETS,
    output_hdf5_path=OUTPUT_HDF5,
    bbox=BBOX,
    granule_name=granule_basename,
)
print(f"Written to: {recon_path}")

## Step 5 — Structure comparison: reconstituted vs original

In [26]:
def dump_structure(path, label):
    """Print HDF5 group/dataset tree with shapes and dtypes."""
    print(f"\n--- {label} ---")
    with h5py.File(path, "r") as f:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  /{name:50s} {str(obj.shape):20s} {obj.dtype}")
            elif isinstance(obj, h5py.Group) and name != "/":
                print(f"  /{name:50s} Group")
        f.visititems(_visit)

dump_structure(recon_path, f"RECONSTITUTED  ({os.path.basename(recon_path)})")
dump_structure(GRANULE_FILE, f"ORIGINAL       ({os.path.basename(GRANULE_FILE)})")


--- RECONSTITUTED  (gmi_local_reconstituted.h5) ---
  /S1                                                 Group
  /S1/Latitude                                        (2983, 221)          float32
  /S1/Longitude                                       (2983, 221)          float32
  /S1/Quality                                         (2983, 221)          int8
  /S1/SCstatus                                        Group
  /S1/SCstatus/FractionalGranuleNumber                (2983,)              float64
  /S1/SCstatus/SCaltitude                             (2983,)              float32
  /S1/SCstatus/SClatitude                             (2983,)              float32
  /S1/SCstatus/SClongitude                            (2983,)              float32
  /S1/SCstatus/SCorientation                          (2983,)              int16
  /S1/ScanTime                                        Group
  /S1/ScanTime/DayOfMonth                             (2983,)              int8
  /S1/ScanTime/DayOfYear    

## Step 6 — SQLite metadata verification

In [27]:
conn = sqlite3.connect(demo.db_path)
rows = conn.execute(
    'SELECT Dataset, COUNT(*) as cnt FROM "PodsMetadata" GROUP BY Dataset ORDER BY Dataset'
).fetchall()
conn.close()

print(f"SQLite DB: {demo.db_path}")
for dataset_name, cnt in rows:
    print(f"  {dataset_name}: {cnt} group(s)")

SQLite DB: /tmp/stare_pods_local/metadata.db
  GMI_S1: 263 group(s)
  GMI_S2: 251 group(s)
